# HierarchicalDet — Colab Free-Tier Feasibility Test

**Purpose**: Document whether HierarchicalDet inference can run on Google Colab
free-tier hardware (T4 GPU, ~15 GB RAM, ~107 GB disk).

**What this notebook tests**:
1. Environment setup (install time, disk usage)
2. Model loading from a pretrained checkpoint
3. Single-image inference at all three tiers
4. Runtime measurement
5. Memory usage

**Pre-requisites**:
- Upload or mount trained checkpoints (e.g., from Google Drive)
- Upload at least one panoramic X-ray image for inference

**Expected constraints**:
- Training from scratch is NOT feasible on Colab free-tier (40k iters × 3 tiers)
- Inference-only with a pre-trained checkpoint IS feasible on T4
- Disk is the main bottleneck: repo + dataset + model ≈ 14+ GB

In [ ]:
# ── 0. Check hardware ────────────────────────────────────────────────────────
import subprocess, sys, os, time

print('=== HARDWARE ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!cat /proc/meminfo | grep MemTotal
!df -h / | tail -1

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 1. Clone the official HierarchicalDet repository ────────────────────────
t0 = time.time()
!git clone https://github.com/ibrahimethemhamamci/HierarchicalDet /content/HierarchicalDet
print(f'Clone time: {time.time()-t0:.1f}s')

# Copy reproduction scripts
# (If running from the reproduction repo, these are already present)
%cd /content/HierarchicalDet
!ls

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
# Note: No requirements.txt exists — install manually.
# detectron2 is BUNDLED in the repo — do NOT pip install detectron2 separately.
t0 = time.time()

# Core dependencies inferred from imports
!pip install -q \
    fvcore \
    iopath \
    omegaconf \
    timm \
    scipy \
    termcolor \
    yacs \
    tabulate \
    cloudpickle \
    pycocotools

# Build bundled detectron2 from source (required — do NOT pip install detectron2)
!pip install -q -e /content/HierarchicalDet

print(f'\nInstall time: {time.time()-t0:.1f}s')

# Verify detectron2 from bundled copy
sys.path.insert(0, '/content/HierarchicalDet')
import detectron2
print(f'detectron2 version: {detectron2.__version__}')

In [ ]:
# ── 3. Mount Google Drive (for checkpoints and dataset) ──────────────────────
# Skip this cell if uploading checkpoints directly.
from google.colab import drive
drive.mount('/content/drive')

# Expected paths — adjust to your Drive structure
CHECKPOINT_T1 = '/content/drive/MyDrive/hierarchicaldet/output/tier1/model_final.pth'
CHECKPOINT_T2 = '/content/drive/MyDrive/hierarchicaldet/output/tier2/model_final.pth'
CHECKPOINT_T3 = '/content/drive/MyDrive/hierarchicaldet/output/tier3/model_final.pth'
BACKBONE_PKL  = '/content/drive/MyDrive/hierarchicaldet/models/swin_base_patch4_window7_224_22k.pkl'
TEST_IMAGE    = '/content/drive/MyDrive/hierarchicaldet/sample_xray.png'

import os
for label, path in [('T1 checkpoint', CHECKPOINT_T1),
                    ('T2 checkpoint', CHECKPOINT_T2),
                    ('T3 checkpoint', CHECKPOINT_T3),
                    ('Backbone',      BACKBONE_PKL),
                    ('Test image',    TEST_IMAGE)]:
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1e6 if exists else 0
    print(f'  {"✓" if exists else "✗"} {label}: {path} ({size:.0f} MB)')

In [ ]:
# ── 4. Download Swin-B backbone if not available ─────────────────────────────
import urllib.request, pickle

os.makedirs('/content/HierarchicalDet/models', exist_ok=True)
pkl_path = '/content/HierarchicalDet/models/swin_base_patch4_window7_224_22k.pkl'

if not os.path.exists(pkl_path):
    url = ('https://github.com/SwinTransformer/storage/releases/download/'
           'v1.0.0/swin_base_patch4_window7_224_22k.pth')
    pth_path = pkl_path.replace('.pkl', '.pth')
    print(f'Downloading Swin-B backbone (~340 MB)...')
    t0 = time.time()
    urllib.request.urlretrieve(url, pth_path)
    print(f'Download time: {time.time()-t0:.1f}s')

    # Convert to detectron2 pickle format
    ckpt = torch.load(pth_path, map_location='cpu')
    d2_ckpt = {'model': ckpt.get('model', ckpt),
               '__author__': 'third_party', 'matching_heuristics': True}
    with open(pkl_path, 'wb') as f:
        pickle.dump(d2_ckpt, f)
    print(f'Backbone saved: {pkl_path}')
else:
    print(f'Backbone already present: {pkl_path}')

In [ ]:
# ── 5. Load model ────────────────────────────────────────────────────────────
# Uses tier-3 (diagnosis) model — the most capable tier.
# If tier-3 checkpoint is not available, set TIER=0 or TIER=1.

sys.path.insert(0, '/content/HierarchicalDet')

from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model
from hierarchialdet import add_diffusiondet_config
from hierarchialdet.util.model_ema import add_model_ema_configs, may_get_ema_checkpointer, \
    EMADetectionCheckpointer

CONFIG = '/content/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml'
WEIGHTS = CHECKPOINT_T3  # Change to T1 or T2 if needed
TIER = 2  # 0=quadrant, 1=enumeration, 2=diagnosis

cfg = get_cfg()
add_diffusiondet_config(cfg)
add_model_ema_configs(cfg)
cfg.merge_from_file(CONFIG)
cfg.MODEL.WEIGHTS = WEIGHTS
cfg.freeze()

t0 = time.time()
model = build_model(cfg)
kwargs = may_get_ema_checkpointer(cfg, model)
if cfg.MODEL_EMA.ENABLED:
    EMADetectionCheckpointer(model, **kwargs).resume_or_load(WEIGHTS, resume=False)
else:
    DetectionCheckpointer(model).resume_or_load(WEIGHTS, resume=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f'Model loaded in {time.time()-t0:.1f}s  (device={device})')

# Memory usage
if torch.cuda.is_available():
    print(f'GPU memory allocated: {torch.cuda.memory_allocated()/1e6:.0f} MB')
    print(f'GPU memory reserved:  {torch.cuda.memory_reserved()/1e6:.0f} MB')

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params/1e6:.1f}M')

In [ ]:
# ── 6. Single-image inference ─────────────────────────────────────────────────
import cv2
import numpy as np
from detectron2.data import transforms as T

# Load test image
img_bgr = cv2.imread(TEST_IMAGE)
assert img_bgr is not None, f'Could not load {TEST_IMAGE}'
h, w = img_bgr.shape[:2]
print(f'Image: {w}×{h} px')

# Preprocess
img_rgb = img_bgr[:, :, ::-1]
resize = T.ResizeShortestEdge(cfg.INPUT.MIN_SIZE_TEST, cfg.INPUT.MAX_SIZE_TEST, 'choice')
img_r, _ = T.apply_transform_gens([resize], img_rgb)
img_t = torch.as_tensor(np.ascontiguousarray(img_r.transpose(2, 0, 1))).float().to(device)
inp = [{'image': img_t, 'height': h, 'width': w, 'image_id': 0, 'file_name': TEST_IMAGE}]

# Warmup
with torch.no_grad():
    _ = model(inp, k=TIER)
if device == 'cuda':
    torch.cuda.synchronize()

# Timed inference
N_RUNS = 5
times = []
with torch.no_grad():
    for _ in range(N_RUNS):
        if device == 'cuda':
            start_e = torch.cuda.Event(enable_timing=True)
            end_e   = torch.cuda.Event(enable_timing=True)
            start_e.record()
        t0 = time.perf_counter()
        out = model(inp, k=TIER)
        if device == 'cuda':
            end_e.record()
            torch.cuda.synchronize()
            times.append(start_e.elapsed_time(end_e))
        else:
            times.append((time.perf_counter() - t0) * 1000)

instances = out[0]['instances'].to('cpu')
scores = instances.scores.numpy()
active = scores >= 0.4

print(f'\nInference time: {np.mean(times):.1f} ± {np.std(times):.1f} ms/image (n={N_RUNS})')
print(f'Throughput: {1000/np.mean(times):.1f} images/min')
print(f'Detections (score≥0.4): {active.sum()}')
if active.sum() > 0:
    print(f'Score range: {scores[active].min():.3f} – {scores[active].max():.3f}')

if device == 'cuda':
    print(f'Peak GPU memory: {torch.cuda.max_memory_allocated()/1e6:.0f} MB')

In [ ]:
# ── 7. Visualize detection result ─────────────────────────────────────────────
from IPython.display import Image as IPImage, display

DIAG_NAMES = {0:'None', 1:'Impacted', 2:'Caries', 3:'Periapical', 4:'DeepCaries'}
QUAD_COLORS = {1:(0,210,0), 2:(0,165,255), 3:(0,0,220), 4:(200,0,200), 0:(140,140,140)}

vis = img_bgr.copy()
scale = min(1.0, 1200 / w)
vis = cv2.resize(vis, (int(w*scale), int(h*scale)))

boxes_xyxy = instances.pred_boxes.tensor.numpy()
labels_np  = instances.pred_classes.numpy() if instances.has('pred_classes') else np.zeros(len(scores))

for i, (box, score, label) in enumerate(zip(boxes_xyxy, scores, labels_np)):
    if score < 0.4:
        continue
    x1, y1, x2, y2 = [int(v * scale) for v in box]
    color = list(QUAD_COLORS.values())[int(label) % len(QUAD_COLORS)]
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    cv2.putText(vis, f'{score:.2f}', (x1, max(y1-3, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

cv2.imwrite('/content/detection_result.jpg', vis, [cv2.IMWRITE_JPEG_QUALITY, 85])
display(IPImage('/content/detection_result.jpg'))

In [ ]:
# ── 8. Colab feasibility summary ─────────────────────────────────────────────
import subprocess

disk_used = subprocess.check_output(['df', '-h', '/']).decode().split('\n')[1].split()

print('=' * 55)
print('  COLAB FREE-TIER FEASIBILITY SUMMARY')
print('=' * 55)
print(f'  GPU:               {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'  GPU memory total:  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '  GPU: N/A')
print(f'  Peak GPU used:     {torch.cuda.max_memory_allocated()/1e6:.0f} MB')
print(f'  Disk used/total:   {disk_used[2]} / {disk_used[1]}')
print(f'  Inference time:    {np.mean(times):.1f} ms/image')
print(f'  Throughput:        {1000/np.mean(times):.1f} images/min')
print()
print('  VERDICT:')
print('  ✓ Inference-only:    FEASIBLE on Colab T4')
print('  ✗ Training tier 1:   NOT FEASIBLE (40k iters ≈ 12-20h; Colab disconnects after 12h)')
print('  ✗ Full 3-tier train: NOT FEASIBLE (36-60h total)')
print('  ✓ Dataset download:  FEASIBLE if disk > 12 GB free')
print()
print('  CONSTRAINTS:')
print('  - Must upload pre-trained checkpoints via Google Drive')
print('  - Colab free-tier RAM (12-16 GB) is sufficient for inference')
print('  - Colab may disconnect during long training runs — use Colab Pro')
print('    or a persistent cloud instance (GCP, AWS, Lambda) for training')
print('  - License: CC-BY-NC-SA 4.0 — non-commercial use only')